# Example: Training YOLOv8 Instance Segmentation

Original tutorial notebook for fine-tuning YOLOv8 on the boulder dataset.
Covers: loading pretrained weights, setting training hyperparameters (200 epochs, `imgsz=1024`, `mask_ratio=1`), and running inference with the trained model.

**Requires GPU.** Runtime: several hours (200 epochs, `batch=4`, `imgsz=1024`).

**Outputs:** Training results + checkpoints written to the `YOLOv8-training/` directory.

*Note: This is the base repo tutorial notebook, not a research analysis notebook.*

In [9]:
from ultralytics import YOLO
from matplotlib import pyplot as plt
from PIL import Image
from pathlib import Path
import yaml
import os
import torch

In [10]:
torch.version.cuda

'11.6'

In [2]:
os.chdir("D:/BOULDERING/data/yolov8/training/")

## Loading of the model
To see other potential models: Check here https://docs.ultralytics.com/tasks/segment/

In [3]:
#Instance
model = YOLO('yolov8n-seg.yaml')  # build a new model from YAML
model = YOLO('yolov8n-seg.pt')  # Transfer the weights from a pretrained model (recommended for training, here from ImageNet)

## Training of the model (incl. specifying train/validation datasets)
For more information on the possible different arguments: see here https://docs.ultralytics.com/modes/train/#arguments. See https://github.com/ultralytics/ultralytics/blob/42744a1717f7e9ccaf2b3ab551332cd09ac24653/ultralytics/cfg/default.yaml#L50 for default config file (with more info on parameters). There are also more hyperparameters for predictions (see config file above). 

In [13]:
results = model.train(data="D:/BOULDERING/data/yolov8/datasets/boulder2024/boulder2024.yaml",
                      project="YOLOv8-training",
                      name="YOLOv8-200-epochs-19-02-2024-1024x1024",
                      save_dir="D:/BOULDERING/data/yolov8/training/", 
                      epochs=200, # (int) number of epochs to train for
                      patience=0, #I am setting patience=0 to disable early stopping.
                      batch=4, # (int) number of images per batch (-1 for AutoBatch)
                      imgsz=1024, # (int | list) input images size as int for train and val modes, or list[w,h] for predict and export modes
                      save=True,
                      save_period=20, # (int) Save checkpoint every x epochs (disabled if < 1)
                      device=0,
                      workers=0,
                      exist_ok=False, # (bool) whether to overwrite existing experiment
                      optimizer="Adam", # wonder if SGD would go faster
                      lr0=0.001,
                      val=True,
                      plots=True,
                      verbose=True,
                      resume=False,
                      cache=False, # (bool) True/ram, disk or False. Use cache for data loading
                      seed=0, # (int) random seed for reproducibility
                      close_mosaic=10, # (int) disable mosaic augmentation for final epochs (0 to disable)
                      overlap_mask=True, # (bool) masks should overlap during training (segment train only)
                      iou=0.7, # (float) intersection over union (IoU) threshold for NMS
                      mask_ratio=1, # I don't want any downsampling, it was actually giving better results with 4? not sure why
                      max_det=2000, # (int) maximum number of detections per image
                      translate=0.1, # image translation (+/- fraction)
                      scale=0.5, # image scale (+/- gain)
                      shear=0.0, # image shear (+/- deg)
                      perspective=0.0, # (float) image perspective (+/- fraction), range 0-0.001
                      flipud=0.0, # image flip up-down (probability)
                      fliplr=0.5, # image flip left-right (probability)
                      mosaic=1.0, # image mosaic (probability)
                      mixup=0.0, # image mixup (probability)
                      copy_paste=0.0, # segment copy-paste (probability)
                      erasing=0.4) # let's upscale from 500 to 1024

New https://pypi.org/project/ultralytics/8.1.15 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.14 🚀 Python-3.9.18 torch-1.13.1+cu116 CUDA:0 (NVIDIA GeForce RTX 4090 Laptop GPU, 16376MiB)
WARNING ⚠️ Upgrade to torch>=2.0.0 for deterministic training.
engine\trainer: task=segment, mode=train, model=yolov8n-seg.pt, data=D:/BOULDERING/data/yolov8/datasets/boulder2024/boulder2024.yaml, epochs=200, time=None, patience=0, batch=4, imgsz=1024, save=True, save_period=20, cache=False, device=0, workers=0, project=YOLOv8-training, name=YOLOv8-200-epochs-19-02-2024-1024x1024, exist_ok=True, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=1, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=2000, half=False, dnn=False, plots=

train: Scanning D:\BOULDERING\data\yolov8\datasets\boulder2024\train\labels.cache... 3957 images, 6 backgrounds, 0 corrupt: 100%|██████████| 3957/3957 [00:00<?, ?it/s]

train: WARNING ⚠️ D:\BOULDERING\data\yolov8\datasets\boulder2024\train\images\M174285569LE_00532_image.png: 1 duplicate labels removed
train: WARNING ⚠️ D:\BOULDERING\data\yolov8\datasets\boulder2024\train\images\M174285569LE_00776_image.png: 1 duplicate labels removed
train: WARNING ⚠️ D:\BOULDERING\data\yolov8\datasets\boulder2024\train\images\M174285569LE_00897_image.png: 1 duplicate labels removed
train: WARNING ⚠️ D:\BOULDERING\data\yolov8\datasets\boulder2024\train\images\M174285569LE_01137_image.png: 1 duplicate labels removed
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))



val: Scanning D:\BOULDERING\data\yolov8\datasets\boulder2024\validation\labels.cache... 739 images, 3 backgrounds, 0 corrupt: 100%|██████████| 739/739 [00:00<?, ?it/s]

Plotting labels to YOLOv8-training\YOLOv8-200-epochs-19-02-2024-1024x1024\labels.jpg... 


optimizer: Adam(lr=0.001, momentum=0.937) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 1024 train, 1024 val
Using 0 dataloader workers
Logging results to YOLOv8-training\YOLOv8-200-epochs-19-02-2024-1024x1024
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      1/200      14.6G       2.08      2.089      1.489      0.976         89       1024: 100%|██████████| 990/990 [11:06<00:00,  1.49it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95):   2%|▏         | 2/93 [00:03<02:10,  wandb: WARNING (User provided step: 1 is less than current step: 3. Dropping entry: {'train/box_loss': 2.08018, 'train/seg_loss': 2.08922, 'train/cls_loss': 1.48938, 'train/dfl_loss': 0.97601, '_timestamp': 1708358758.735709}).
wandb: WARNING (User provided step: 1 is less than current step: 3. Dropping entry: {'lr/pg0': 0.06703333333333333, 'lr/pg1': 0.00033299663299663304, 'lr/pg2': 0.00033299663299663304, '_timestamp': 1708358758.735709}).
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 93/93 [01:57<00:00, 


                   all        739      26830      0.621      0.555      0.615      0.256      0.457      0.377      0.353      0.104

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


      2/200      4.38G      1.961      1.941       1.54     0.9783        331       1024:   1%|          | 7/990 [00:05<15:38,  1.05it/s]wandb: WARNING (User provided step: 1 is less than current step: 3. Dropping entry: {'metrics/precision(B)': 0.62094, 'metrics/recall(B)': 0.5555, 'metrics/mAP50(B)': 0.61478, 'metrics/mAP50-95(B)': 0.25627, 'metrics/precision(M)': 0.45692, 'metrics/recall(M)': 0.37684, 'metrics/mAP50(M)': 0.3531, 'metrics/mAP50-95(M)': 0.10379, 'val/box_loss': 1.93409, 'val/seg_loss': 1.94992, 'val/cls_loss': 1.30917, 'val/dfl_loss': 0.93778, '_timestamp': 1708358877.184579}).
wandb: WARNING (User provided step: 1 is less than current step: 3. Dropping entry: {'labels': {'_type': 'image-file', 'sha256': '8f8fd74f08bdb28037150838b59f10c0f56833f5a735b6e8654fafeb1d6913ac', 'size': 114480, 'path': 'media/images/labels_3_8f8fd74f08bdb2803715.jpg', 'format': 'jpg', 'width': 1600, 'height': 1600}, '_timestamp': 1708358877.1915689}).
wandb: WARNING (User provided step: 1 is 

KeyboardInterrupt: 

I feel like there is a huge difference on how long it takes to go through an epoch. Sometimes it takes about 10 min, and then, some other times it is showing up to one hour. I am not 100% sure why this is the case... The re-scaling of the image should happen at the batch level, so I do not really understand why it would be such a large variations. 

## Inference

Now that our model is trained, we can use it for inference. You can load the best model or the latest. I am picking the latest.

In [ ]:
my_new_model = YOLO('/content/drive/MyDrive/ColabNotebooks/data/3D-EM-Platelet/YOLOV8-data/results/200_epochs-4/weights/last.pt')
new_results = my_new_model.predict(new_image, conf=0.2)

The results are stored in a variable 'new_results'. Since we only have one image for segmentation, we will only have one set of results. Therefore, let us work with that one result.

In [ ]:
new_result_array = new_results[0].plot()
plt.figure(figsize=(12, 12))
plt.imshow(new_result_array)

### How to extract bounding boxes and segmented masks from the result

In [ ]:
new_result = new_results[0]
new_result

#### bounding boxes
See https://github.com/bnsreenu/python_for_microscopists/blob/master/334_training_YOLO_V8_EM_platelets_converted_labels.ipynb if you want to extract masks per classes. Also there is an interesting script to look at region properties (regionprops from skimage) of masks. 

In [ ]:
class_names = new_result.names.values()
class_names

In [ ]:
# Extract the boxes, which likely contain class IDs
detected_boxes = new_result.boxes.data
# Extract class IDs from the detected boxes
class_labels = detected_boxes[:, -1].int().tolist()
# Initialize a dictionary to hold masks by class
masks_by_class = {name: [] for name in new_result.names.values()}

# Iterate through the masks and class labels
for mask, class_id in zip(extracted_masks, class_labels):
    class_name = new_result.names[class_id]  # Map class ID to class name
    masks_by_class[class_name].append(mask.cpu().numpy())

#### mask

In [ ]:
extracted_masks = new_result.masks.data
extracted_masks.shape

In [ ]:
masks_array = extracted_masks.cpu().numpy()

In [ ]:
plt.imshow(masks_array[9])

## Export model to ONNX for deployment

In [ ]:
my_new_model.export(format='onnx', imgsz=[1024,1024]) # do you have to give images with a specific size?